# 04 - Posting time heatmap

Notebook xac dinh khung gio/ngay co engagement rate va engagement cao nhat tu export heatmap.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

heatmap = read_csv('dashboard/exports/vw_posting_time_heatmap.csv')
posts = read_csv('data/processed/posts.csv')


In [ ]:
top_rate_windows = heatmap.dropna(subset=['avg_engagement_rate']).sort_values(['avg_engagement_rate', 'total_engagement'], ascending=False).head(10)
top_volume_windows = heatmap.sort_values(['total_engagement', 'post_count'], ascending=False).head(10)
display(top_rate_windows)
display(top_volume_windows)


In [ ]:
pivot_rate = heatmap.pivot_table(index='day_name', columns='hour_of_day', values='avg_engagement_rate', aggfunc='mean')
pivot_posts = heatmap.pivot_table(index='day_name', columns='hour_of_day', values='post_count', aggfunc='sum', fill_value=0)
display(pivot_rate.round(2))
display(pivot_posts.astype(int))


In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
by_day = heatmap.groupby('day_name', as_index=False).agg(
    post_count=('post_count', 'sum'),
    total_engagement=('total_engagement', 'sum'),
    avg_engagement_rate=('avg_engagement_rate', 'mean'),
)
by_day['day_name'] = pd.Categorical(by_day['day_name'], categories=day_order, ordered=True)
display(by_day.sort_values('day_name').round(4))


## Insight

- Key finding: Slot co ER tot nhat va engagement volume cao nhat hien cung la Wednesday 12:00.
- Supporting data: Friday 07:00 dat 3.4456% ER; Saturday 07:00 dat 3.0822% ER; Thursday 20:00 co 1,145 engagements va 3.0207% ER.
- Recommended action: Uu tien test Wednesday noon cho ca chat luong tuong tac va volume, nhung can them mau vi moi slot top chi co 1 post.